In [1]:
from pathlib import Path
import time
import requests
import pandas as pd 

In [2]:
RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True) 

In [5]:
URL_ECO2MIX = ("https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
               "eco2mix-national-cons-def/exports/csv")
params = {"select": "date_heure,consommation", "delimiter": ",", "order_by": "date_heure"}

r = requests.get(URL_ECO2MIX, params=params, timeout=300)
r.raise_for_status()
(RAW / "eco2mix.csv").write_bytes(r.content)

15685324

In [7]:
conso = pd.read_csv(RAW / "eco2mix.csv", encoding="utf-8-sig")
print(conso.shape)
conso.head(10)

(514272, 2)


,date_heure,consommation
0,2011-12-31T23:00:00+00:00,58315.0
1,2011-12-31T23:15:00+00:00,NaN
2,2011-12-31T23:30:00+00:00,58315.0
3,2011-12-31T23:45:00+00:00,NaN
4,2012-01-01T00:00:00+00:00,56231.0
5,2012-01-01T00:15:00+00:00,NaN
6,2012-01-01T00:30:00+00:00,56075.0
7,2012-01-01T00:45:00+00:00,NaN
8,2012-01-01T01:00:00+00:00,55532.0
9,2012-01-01T01:15:00+00:00,NaN


In [9]:
VILLES = {
    "Paris": (48.8566, 2.3522), "Lyon": (45.7640, 4.8357), "Marseille": (43.2965, 5.3698), "Toulouse": (43.6047, 1.4442),
    "Nice": (43.7102, 7.2620), "Nantes": (47.2184, -1.5536), "Strasbourg": (48.5734, 7.7521), "Montpellier": (43.6119, 3.8772),
    "Bordeaux": (44.8378, -0.5792), "Lille": (50.6292, 3.0573), "Rennes": (48.1173, -1.6778), "Reims": (49.2583, 4.0317), 
    "Le Havre": (49.4944, 0.1079), "Saint-Étienne": (45.4397, 4.3872), "Toulon": (43.1242, 5.9280), "Grenoble": (45.1885, 5.7245),
    "Dijon": (47.3220, 5.0415), "Angers": (47.4784, -0.5632), "Nîmes": (43.8367, 4.3601),
}
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"

frames = []
for ville, (lat, lon) in VILLES.items():
    r = requests.get(URL_METEO, params={
        "latitude": lat,
        "longitude": lon,
        "start_date": "2012-01-01",
        "end_date": "2026-08-31",
        "hourly": "temperature_2m",
        "timezone" : "UTC",
    }, timeout=300)
    r.raise_for_status()
    h = r.json()["hourly"]
    frames.append(pd.DataFrame({"time": h["time"], "temperature_2m": h["temperature_2m"], "ville": ville}))
    print(f"{ville} OK")
    time.sleep(1)

meteo = pd.concat(frames, axis=1)
meteo.index.name = "time"
meteo.to_csv(RAW / "meteo.csv")
print(meteo.shape)
meteo.head()

Paris OK
Lyon OK
Marseille OK
Toulouse OK
Nice OK
Nantes OK
Strasbourg OK
Montpellier OK
Bordeaux OK
Lille OK
Rennes OK
Reims OK


HTTPError: 429 Client Error: Too Many Requests for url: https://archive-api.open-meteo.com/v1/archive?latitude=49.4944&longitude=0.1079&start_date=2012-01-01&end_date=2026-08-31&hourly=temperature_2m&timezone=UTC

In [11]:
RAW_METEO = RAW / "meteo"
RAW_METEO.mkdir(exist_ok=True)

for df in frames:
    v = df["ville"].iloc[0]
    s = pd.Series(df["temperature_2m"].values, index=pd.to_datetime(df["time"]), name=v)
    s.to_csv(RAW_METEO / f"{v}.csv", index_label="time")
print(len(frames), "villes sauvegardées")

12 villes sauvegardées


In [12]:
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"
RAW_METEO = RAW / "meteo"
RAW_METEO.mkdir(exist_ok=True)

def telecharger_ville(ville, lat, lon, essais=5):
    for essai in range(essais):
        r = requests.get(URL_METEO, params={
            "latitude": lat, "longitude": lon,
            "start_date": "2012-01-01", "end_date": "2026-08-31",
            "hourly": "temperature_2m", "timezone": "UTC",
        }, timeout=300)
        if r.status_code == 429:
            attente = 60 * (essai + 1)
            print(f"{ville} : quota atteint, nouvel essai dans {attente}s")
            time.sleep(attente)
            continue
        r.raise_for_status()
        h = r.json()["hourly"]
        return pd.Series(h["temperature_2m"], index=pd.to_datetime(h["time"]), name=ville)
    raise RuntimeError(f"{ville} : toujours bloqué, relance la cellule plus tard")

for ville, (lat, lon) in VILLES.items():
    fichier = RAW_METEO / f"{ville}.csv"
    if fichier.exists():
        print(f"{ville} déjà téléchargé")
        continue
    telecharger_ville(ville, lat, lon).to_csv(fichier, index_label="time")
    print(f"{ville} OK")
    time.sleep(1)


Paris déjà téléchargé
Lyon déjà téléchargé
Marseille déjà téléchargé
Toulouse déjà téléchargé
Nice déjà téléchargé
Nantes déjà téléchargé
Strasbourg déjà téléchargé
Montpellier déjà téléchargé
Bordeaux déjà téléchargé
Lille déjà téléchargé
Rennes déjà téléchargé
Reims déjà téléchargé
Le Havre OK
Saint-Étienne OK
Toulon OK
Grenoble OK
Dijon OK
Angers OK
Nîmes OK


In [14]:
meteo = pd.concat(
    [pd.read_csv(RAW_METEO / f"{v}.csv", index_col="time", parse_dates=True)[v] for v in VILLES],
    axis=1,
)
meteo.to_csv(RAW / "meteo.csv")
print(meteo.shape)
meteo.head()

(128568, 19)


,Paris,Lyon,Marseille,Toulouse,Nice,Nantes,Strasbourg,Montpellier,Bordeaux,Lille,Rennes,Reims,Le Havre,Saint-Étienne,Toulon,Grenoble,Dijon,Angers,Nîmes
time,,,,,,,,,,,,,,,,,,,
2012-01-01 00:00:00,12.1,8.5,11.7,11.2,8.1,12.3,9.1,13.2,12.6,11.1,12.0,11.3,11.9,9.2,12.4,8.5,10.1,11.8,10.8
2012-01-01 01:00:00,12.1,8.3,11.5,11.2,7.4,12.1,9.1,13.3,12.5,11.0,12.1,11.4,11.9,8.8,12.1,8.4,10.2,11.9,10.6
2012-01-01 02:00:00,12.0,7.9,11.4,11.2,7.0,12.3,9.2,13.2,12.5,11.2,12.2,11.5,11.8,8.4,12.0,7.9,10.2,12.0,10.4
2012-01-01 03:00:00,11.8,7.5,11.2,11.2,7.4,12.4,9.3,13.0,12.5,11.4,12.3,11.4,11.9,8.3,11.9,7.7,10.1,12.0,10.1
2012-01-01 04:00:00,11.5,7.8,10.9,11.3,7.6,12.4,9.4,12.7,12.4,11.5,12.3,11.3,11.9,8.3,11.8,8.0,10.0,12.1,9.8
